In [102]:
import re
import pandas as pd

## 1. Read and parse text file

In [103]:
def parse_real_estate_data(raw_data: str) -> pd.DataFrame:
    '''
    Turn real_estate.txt into a dataframe

    Parameters:
        raw_data: content string of real_estate.txt

    Returns:
        pd.DataFrame containing extracted records
    '''

    record_blocks = re.split(r'\n-+\n', raw_data.strip())

    all_records = []
    
    for block in record_blocks:
        if not block.strip():
            continue

        record = {}
        lines = block.strip().split('\n')
        
        for line in lines:
            if ':' in line:
                # Split at the first colon only to preserve full url & desc
                parts = line.split(':', maxsplit=1)

                key = parts[0].strip()
                value = parts[1].strip()
                
                record[key] = value

        all_records.append(record)

    df = pd.DataFrame(all_records)

    return df

In [104]:
with open('../data/raw/real_estate.txt', encoding='utf-8') as file:
    raw = file.read() 

df = parse_real_estate_data(raw)
df.head()

,ID,URL,Title,Price,Area,Bedrooms,Bathrooms,Legal,Interior,Facing Direction,Balcony Direction,Front Width,Front Road Width,Description,Verified,Location,Scraped At,LH
0,1,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,"Vô cùng hối tiếc khi không mua, The Gió Rivers...","2,47 tỷ",65 m²,2 phòng,2 phòng,Hợp đồng mua bán,Đầy đủ,None,None,None,None,"The Gió Riverside của CĐT An Gia, chính thức n...",Yes,"Dự án The Gió Riverside, Đường Vành Đai 3, Phư...",2025-10-05 06:29:49,NaN
1,2,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,The Gió - Thanh toán tối đa chỉ 450tr trong 3 ...,Thỏa thuận,"65,1 m²",2 phòng,2 phòng,Hợp đồng mua bán,Cơ bản,None,None,None,None,Booking sớm chọn giỏ hàng đẹp căn hộ The Gió R...,Yes,"Dự án The Gió Riverside, Đường ĐT 16, Phường B...",2025-10-05 06:29:59,NaN
2,3,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,BQL Vinhomes Smart City cập nhật quỹ căn: Stud...,"4,6 tỷ","64,6 m²",2 phòng,2 phòng,Sổ đỏ/ Sổ hồng.,Đầy đủ.,Tây - Bắc,Đông - Nam,None,None,Em Hoàng Giang là cư dân sống tại Vinhomes Sm...,Yes,"Dự án The Sapphire-Vinhomes Smart City, Phường...",2025-10-05 06:30:08,NaN
3,4,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,"Chỉ cần 200tr, trả góp 4tr5/tháng sở hữu nhà n...","1,89 tỷ","55,3 m²",1 phòng,1 phòng,None,Cơ bản. Nội thất: NT cơ bản đến từ các thương ...,None,Tây - Nam,None,None,Em xin gửi đến anh chị thông tin tổng hợp dự á...,Yes,"Dự án Phú Đông SkyOne, Đường ĐT 743C, Phường T...",2025-10-05 06:30:20,NaN
4,5,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,"Giá thật! CK cao nhất 10/2025, trực tiếp CĐT 1...","3,57 tỷ","52,6 m²",1 phòng,1 phòng,Hợp đồng mua bán.,None,None,None,None,None,"Chính sách mới của CĐT Gamuda Land 10/2025, mở...",Yes,"Dự án Elysian, Đường Lò Lu, Phường Trường Thạn...",2025-10-05 06:30:31,NaN


In [105]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ID                 2722 non-null   object
 1   URL                2722 non-null   object
 2   Title              2722 non-null   object
 3   Price              2722 non-null   object
 4   Area               2722 non-null   object
 5   Bedrooms           2722 non-null   object
 6   Bathrooms          2722 non-null   object
 7   Legal              2722 non-null   object
 8   Interior           2722 non-null   object
 9   Facing Direction   2722 non-null   object
 10  Balcony Direction  2722 non-null   object
 11  Front Width        2722 non-null   object
 12  Front Road Width   2722 non-null   object
 13  Description        2722 non-null   object
 14  Verified           2722 non-null   object
 15  Location           2722 non-null   object
 16  Scraped At         2722 non-null   object


## 2. Basic cleanup

### &emsp; 2.1. Extract values

#### &emsp; &emsp; 2.1.1. Extract numeric values

In [106]:
def extract_numeric(s:str|None) -> float:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^(\d+.?\d*,?\d*)\D?", s)
    if not results:
        return None
    else:
        num_str = results.group(1)
        num_str = num_str.replace(".","")
        num_str = num_str.replace(",",".")
        return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+.?\d*,?\d*\s*(\D*)", s)
    if not results:
        return None
    else:
        return results.group(1)

In [107]:
df["price_val"] = df["Price"].apply(extract_numeric)
df["price_unit"] = df["Price"].apply(extract_measuring_unit)
df["area_val"] = df["Area"].apply(extract_numeric)
df["area_unit"] = df["Area"].apply(extract_measuring_unit)
df["n_bedrooms"] = df["Bedrooms"].apply(extract_numeric)
df["n_bathrooms"] = df["Bathrooms"].apply(extract_numeric)
df["front_width_val"] = df["Front Width"].apply(extract_numeric)
df["front_width_unit"] = df["Front Width"].apply(extract_measuring_unit)
df["front_road_width_val"] = df["Front Road Width"].apply(extract_numeric)
df["front_road_width_unit"] = df["Front Road Width"].apply(extract_measuring_unit)

df[[
    "price_val", "price_unit", "area_val", "area_unit", "n_bedrooms", "n_bathrooms", 
    "front_width_val", "front_width_unit", "front_road_width_val", "front_road_width_unit"
]].describe(include="all")

,price_val,price_unit,area_val,area_unit,n_bedrooms,n_bathrooms,front_width_val,front_width_unit,front_road_width_val,front_road_width_unit
count,2383.00000,2383,2722.000000,2722,1748.000000,1652.000000,1133.000000,1133,992.000000,992
unique,NaN,5,NaN,1,NaN,NaN,NaN,1,NaN,1
top,NaN,tỷ,NaN,m²,NaN,NaN,NaN,m,NaN,m
freq,NaN,2084,NaN,2722,NaN,NaN,NaN,1133,NaN,992
mean,35.01859,NaN,329.568916,NaN,3.486842,3.234867,9.211192,NaN,13.690524,NaN
std,102.14035,NaN,2546.525614,NaN,4.580058,4.732382,13.567192,NaN,12.586475,NaN
min,1.00000,NaN,13.000000,NaN,1.000000,1.000000,1.000000,NaN,1.000000,NaN
25%,4.20000,NaN,62.500000,NaN,2.000000,2.000000,5.000000,NaN,6.000000,NaN
50%,9.40000,NaN,88.000000,NaN,3.000000,2.000000,6.000000,NaN,11.500000,NaN
75%,23.45000,NaN,140.000000,NaN,4.000000,4.000000,10.000000,NaN,17.000000,NaN


#### &emsp;&emsp; 2.1.3. Extract `property_type` from `URL`

In [108]:
PROPERTY_LINKS_TYPES = {
    "ban-can-ho-chung-cu-mini":"Căn hộ chung cư mini",
    "ban-can-ho-chung-cu": "Căn hộ chung cư",
    "ban-nha-rieng": "Nhà riêng",
    "ban-nha-biet-thu-lien-ke": "Nhà biệt thự, liền kề",
    "ban-nha-mat-pho": "Nhà mặt phố",
    "ban-shophouse-nha-pho-thuong-mai": "Shophouse, nhà phố thương mại",
    "ban-dat-nen-du-an": "Đất nền dự án",
    "ban-dat": "Đất",
    "ban-condotel": "Condotel",
    "ban-trang-trai-khu-nghi-duong": "Trang trại, khu nghỉ dưỡng",
    "ban-kho-nha-xuong": "Kho, nhà xưởng",
    "ban-loai-bat-dong-san-khac": "Khác"
}

def extract_propterty_type(url:str) -> str:
    results = re.search(r'^https://batdongsan.com.vn/(.*)/', url)

    if not results:
        return None
    
    type_n_place = results.group(1)

    for type_link_affix in PROPERTY_LINKS_TYPES:
        if type_link_affix in type_n_place:
            return PROPERTY_LINKS_TYPES[type_link_affix]
        
    return None

def shorten_url(url:str) -> str:
    return url[26:]

In [109]:
df['property_type'] = df.URL.apply(extract_propterty_type)
df['short_url'] = df.URL.apply(shorten_url)
df[['property_type', 'short_url']].sample(5)

,property_type,short_url
885,"Nhà biệt thự, liền kề",ban-nha-biet-thu-lien-ke-duong-nguyen-huu-tho-...
2707,Nhà riêng,ban-nha-rieng-phuong-vinh-tuy/chu-giam-sap-ham...
802,Nhà riêng,ban-nha-rieng-duong-quoc-huong-phuong-thao-die...
2189,Đất,ban-dat-xa-van-canh-1/chinh-chu-gui-ban-3-lo-5...
1442,"Shophouse, nhà phố thương mại",ban-shophouse-nha-pho-thuong-mai-xa-vinh-tan-p...


#### &emsp;&emsp; 2.1.4. Extract `city` from `Location`

In [110]:
def extract_location_detail(addr: str, level:int) -> str|None:
    '''
    level:int - the level of details to extract from the address. 1 -> Province; 2 -> District
    '''
    if not isinstance(addr, str) or not addr.strip():
        return None
    parts = [p.strip() for p in addr.split(",") if p.strip() != ""]
    if len(parts) >= level:
        result = parts[-level].replace('.','')
        return result
    return None

In [111]:
df['city_province'] = df.Location.apply(extract_location_detail, level = 1)
df['district'] = df.Location.apply(extract_location_detail, level = 2)
df[['Location', 'city_province', 'district']].sample(5)

,Location,city_province,district
1069,"Hà Đô Charm Villas, Đại lộ Thăng Long, An Thượ...",Hà Nội,Hoài Đức
2329,"Dự án Thanh Bình Residence, Đường Nguyễn Du, P...",Bình Dương,Thuận An
1935,"Dự án Zeitgeist City, Đường Nguyễn Hữu Thọ, Xã...",Hồ Chí Minh,Nhà Bè
995,"Đường Gò Mây, Xã Diên Phước, Diên Khánh, Khánh...",Khánh Hòa,Diên Khánh
5,"Dự án The Gió Riverside, Đường Vành Đai 3, Phư...",Bình Dương,Dĩ An


#### Fix null cities & Standardize city labels

In [120]:
df.city_province.unique()

array(['Bình Dương', 'Hà Nội', 'Hồ Chí Minh', 'Đà Nẵng', 'Quảng Nam',
       'Hưng Yên', 'TP.HCM', 'Bắc Giang', 'Khánh Hòa', 'Quảng Ninh',
       'Vĩnh Phúc', 'Thành phố Hà Nội', 'Long An', 'Hải Phòng',
       'Lâm Đồng', 'Bà Rịa Vũng Tàu', 'Nghệ An', 'Bắc Ninh', 'Nam Định',
       'Quảng Ngãi', 'Đồng Nai', 'Hà Nam', 'Thừa Thiên Huế', 'Thanh Hóa',
       'Bình Phước', 'Thành phố Hồ Chí Minh', 'Hòa Bình', 'Thái Nguyên',
       'Tây Ninh', 'Bạc Liêu', 'Hậu Giang', 'Bình Định', 'Hải Dương',
       'Cần Thơ', 'Bình Thuận', 'Bến Tre', 'Thái Bình', 'Phú Thọ',
       'Kiên Giang', 'Ninh Bình', 'Ninh Thuận', 'thành phố Hồ Chí Minh',
       'Tiền Giang', 'Quảng Trị', 'Quảng Bình'], dtype=object)

In [113]:
df.loc[df.city_province == '86Tỷ', ['city_province', 'district']] = ('TP.HCM', 'Quận Bình Thạnh')
df.loc[1684, ['city_province', 'district']] = ('TP.HCM', 'Huyện Bình Chánh')
df.loc[2192, ['city_province', 'district']] = ('Hà Nội', 'Quận Hà Đông')
df.loc[2561, ['city_province', 'district']] = ('Hưng Yên', 'Văn Lâm')

#### &emsp;&emsp; 2.1.5. Convert "None" strings to `None`

In [114]:
df.replace("None", None, inplace=True)

### &emsp; 2.2. Select columns for merging and analysis

In [115]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     2722 non-null   object 
 1   URL                    2722 non-null   object 
 2   Title                  2722 non-null   object 
 3   Price                  2722 non-null   object 
 4   Area                   2722 non-null   object 
 5   Bedrooms               1748 non-null   object 
 6   Bathrooms              1652 non-null   object 
 7   Legal                  2327 non-null   object 
 8   Interior               1431 non-null   object 
 9   Facing Direction       1177 non-null   object 
 10  Balcony Direction      822 non-null    object 
 11  Front Width            1133 non-null   object 
 12  Front Road Width       992 non-null    object 
 13  Description            2721 non-null   object 
 14  Verified               2722 non-null   object 
 15  Loca

In [116]:
cols_for_analysis = [
    'price_val', 'price_unit', 'area_val', 'area_unit', 'n_bedrooms', 'n_bathrooms', 'front_width_val', 'front_width_unit', 'front_road_width_val', 'front_road_width_unit',
    'Legal', 'Facing Direction', 'Balcony Direction', 'property_type', 'Location', 'city_province', 'district',
    # misc
    'URL', 'Title', 'Description', 'Interior'
]
df_pruned = df[cols_for_analysis]
rename_map = {
    'price_val': 'price',
    'area_val': 'area',
    'front_width_val': 'front_width', 
    'front_road_width_val': 'front_road_width', 
    'Legal':'legal_docs',
    'Facing Direction': 'facing_direction',
    'Balcony Direction': 'balcony_direction',
    'Location': 'address',
    'URL': 'url',
    'Title': 'title',
    'Description': 'description',
    'Interior': 'interior'
}
df_pruned = df_pruned.rename(rename_map, axis=1)
df_pruned.sample(5)

,price,price_unit,area,area_unit,n_bedrooms,n_bathrooms,front_width,front_width_unit,front_road_width,front_road_width_unit,...,facing_direction,balcony_direction,property_type,address,city_province,district,url,title,description,interior
1605,2.90,tỷ,50.0,m²,2.0,2.0,NaN,None,NaN,None,...,Bắc,Nam,Căn hộ chung cư,"Dự án BV Diamond Hill Thái Nguyên, Đường Bắc S...",Thái Nguyên,Thái Nguyên,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,Tặng 50 triêu khi khách hàng mua căn hộ 1 phòn...,Chạm đỉnh sống sang - Nâng hạng vị thế.Quà tặn...,Cơ bản
2273,16.80,tỷ,90.0,m²,4.0,3.0,14.0,m,14.0,m,...,Đông - Nam,Đông - Nam,"Nhà biệt thự, liền kề","The Classia, Đường Võ Chí Công, Phường Phú Hữu...",Hồ Chí Minh,Quận 9,https://batdongsan.com.vn/ban-nha-biet-thu-lie...,"Bán gấp nhà phố 4 tầng Classia Khang Điền 16,8...",Giỏ hàng dự án Classia Khang Điền- Nhà Phố liề...,Không nội thất
820,NaN,None,125.0,m²,2.0,2.0,NaN,None,NaN,None,...,None,Đông - Nam,Căn hộ chung cư,"Hanoi Signature, 6, Đường Nguyễn Văn Huyên, Ph...",Hà Nội,Cầu Giấy,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,Bán căn 2 - 4 PN tại chung cư Hanoi Signature ...,Hanoi Signature dấu ấn sống thượng lưu giữa tr...,Không nội thất
2041,36.00,tỷ,340.0,m²,2.0,2.0,NaN,None,NaN,None,...,None,None,Nhà riêng,"Phường Tân Hưng, Quận 7, Hồ Chí Minh",Hồ Chí Minh,Quận 7,https://batdongsan.com.vn/ban-nha-rieng-phuong...,"Quận 7 , Bán nhà Cạnh Lottemart , DT 340m2 , H...","khu hiện hữu, sổ riêng để nhà , giao dịch ngay...",None
752,2.75,tỷ,74.0,m²,2.0,2.0,NaN,None,NaN,None,...,None,None,Căn hộ chung cư,"Dự án The Rivana, Đường Quốc Lộ 13, Phường Vĩn...",Bình Dương,Thuận An,https://batdongsan.com.vn/ban-can-ho-chung-cu-...,Muốn mua The Rivana giá tốt nhất liên hệ em fu...,Em hỗ trợ miễn phí các dịch vụ liên quan tới c...,Đầy đủ.


## 3. Check and export

In [117]:
df_pruned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   price                  2383 non-null   float64
 1   price_unit             2383 non-null   object 
 2   area                   2722 non-null   float64
 3   area_unit              2722 non-null   object 
 4   n_bedrooms             1748 non-null   float64
 5   n_bathrooms            1652 non-null   float64
 6   front_width            1133 non-null   float64
 7   front_width_unit       1133 non-null   object 
 8   front_road_width       992 non-null    float64
 9   front_road_width_unit  992 non-null    object 
 10  legal_docs             2327 non-null   object 
 11  facing_direction       1177 non-null   object 
 12  balcony_direction      822 non-null    object 
 13  property_type          2722 non-null   object 
 14  address                2722 non-null   object 
 15  city

In [118]:
df_pruned.to_csv('../data/interim/batdongsan_com_vn(2).csv')

In [119]:
df_pruned.area_unit.unique()

array(['m²'], dtype=object)